# v22 OOF Patch v2 — append-only instrumentation

Turns a v22 fork into a leak-free out-of-fold scorer over the training wells and
emits `v22_oof.pkl` for the join against the BiGRU.

**Append these cells to the end of a throwaway v22 fork.** Run v22 top-to-bottom
first, then P0 → P6 in order. Do not run v22's submission cell in this fork.

## What changed from v1

Merged in from the alternate design:

- **`field_aug` guard** (P0) — hard-asserts `CONFIG['field_aug']` is False. The
  patched query path drops the `field_aug` branch as dead code; if that flag were
  ever True, the patch would silently diverge from v22 instead of stopping.
- **`field_conf` and `status` captured** (P6) — `field_conf` is a third routing
  candidate alongside spread and `nn_dist`, and `status` tells you which code path
  each well took, which is how you find out whether the failures cluster.
- **`_UFIELD['pts']` accepted** (P2) — the point array is read from the dict if
  present, falling back to the `_UFIELD_PTS` global. Works either way.

Deliberately **not** merged: the full literal redefinition of `build_ufield`.
That literal includes the `tops_all` / datum-offset head, which cannot be verified
against your actual v22 from here. If it has drifted by a line, redefining the
function wholesale would quietly change v22's datum alignment and the OOF would be
measuring a different model than the one on the leaderboard — while still running
clean and still printing something near 6.5. P1 does verified source surgery
instead: four anchors, each asserted to match exactly once, and it aborts printing
your real source if any anchor misses.

## Run order

| Cell | Purpose | Time |
|---|---|---|
| P0 | Preflight — globals, `field_aug` guard, source hashes | instant |
| P1 | Patch `build_ufield` source: tag each field point with its well | instant |
| P2 | Redefine `_field_query`: exclude-then-build self-exclusion | instant |
| P3 | Wrap `field_blend` + `predict_well_diag` | instant |
| P4 | Rebuild the field so the `wid` tags take effect | ~40 s |
| P5 | **Validation gate** — prove self-exclusion is firing | ~10 s |
| P6 | OOF loop → `v22_oof.pkl` | 45–75 min |

## The two gates that decide whether the output is trustworthy

1. **P5** must show nearest-neighbour distance jumping from a couple of feet to
   hundreds, and field-only blind MAE rising from ~0.2–0.4 (leaked) to roughly
   13–15 (real), while staying finite. If it doesn't move, exclusion isn't firing.
2. **P6** must land near **~6.5**, v22's known local number. Materially above
   means the field blend got switched off; materially below means a leak survived.

In [ ]:
# ===== P0: preflight =====
# Asserts the v22 globals this patch depends on, checks field_aug, and snapshots
# source hashes so the patch is auditable. Nothing is modified here.
import hashlib, inspect, sys
import numpy as np

_REQUIRED = ['build_ufield', '_field_query', 'field_blend',
             'predict_well_diag', 'load_well', 'wells', 'CONFIG']
_missing = [n for n in _REQUIRED if n not in globals()]
if _missing:
    raise NameError(
        'v22 globals not found: %s\n'
        'Run your v22 notebook top-to-bottom FIRST (model-code cell + the '
        'spatial-map/field cell), then run these patch cells.' % _missing)

# --- field_aug guard --------------------------------------------------------
# P2 drops the field_aug branch from the query path as dead code. That is only
# safe while the flag is off. Fail loudly rather than diverge silently.
if CONFIG.get('field_aug', False):
    raise RuntimeError(
        'CONFIG["field_aug"] is True. The patched _field_query in P2 omits the '
        'field_aug branch (LB-falsified, assumed off). Either set it False for '
        'this analysis run, or restore that branch in P2 before proceeding.')
print('field_aug guard OK (False)')

def _srchash(fn):
    try:
        return hashlib.sha256(inspect.getsource(fn).encode()).hexdigest()[:16]
    except Exception:
        return '<unavailable>'

_PRE = {n: _srchash(globals()[n]) for n in _REQUIRED if callable(globals()[n])}
print('\nPREFLIGHT OK — all v22 globals present')
for k, v in _PRE.items():
    print('  %-20s src sha256[:16] = %s' % (k, v))

# --- does v22 already thread self_well into _field_query? --------------------
# If it does, the _CUR_WELL wrapper in P3 is redundant belt-and-braces (harmless:
# an explicit self_well argument always wins). If it does not, _CUR_WELL is the
# only thing making exclusion fire. Either way P5 is the real arbiter.
try:
    _pwd_src = inspect.getsource(predict_well_diag)
    _threads = 'self_well' in _pwd_src
    print('\npredict_well_diag passes self_well explicitly:', _threads)
    if not _threads:
        print('  -> _CUR_WELL plumbing (P3) is load-bearing. P5 will confirm.')
except Exception:
    print('\ncould not inspect predict_well_diag source (not fatal)')

_PATCH_APPLIED = globals().get('_PATCH_APPLIED', set())
print('\npatches already applied in this session:', sorted(_PATCH_APPLIED) or 'none')


## P1 — tag each field point with its well

`build_ufield` builds `wid = [''] * n_train`, so there is no way to tell which
field point came from which well and self-exclusion is impossible. This rewrites
four anchors in the function's **source text**, leaving the datum/`offs` head
untouched:

- `pts = []; us = []` → also open a `widp` list
- the `pts.append(...)` line → tag every point with its well id
- `wid = [''] * n_train` → `wid = list(widp)`
- after `P = np.vstack(pts)` → stash the raw point array for P2's tree rebuilds

Every replacement asserts `count == 1`, and the patched source is `ast.parse`d
before it's exec'd. The full patched source is printed so you can eyeball the
diff. If an anchor misses, the cell prints your real source and stops having
modified nothing — send me that dump and the anchors get re-cut in two minutes.

In [ ]:
# ===== P1: patch build_ufield source (verified, exactly-once replacements) =====
import inspect, hashlib, textwrap, ast

_src0 = inspect.getsource(build_ufield)
print('build_ufield BEFORE sha256 =', hashlib.sha256(_src0.encode()).hexdigest())

_src = textwrap.dedent(_src0)

_ANCHORS = [
    ('A1', "pts = []; us = []",
           "pts = []; us = []; widp = []"),
    ('A2', "pts.append(np.column_stack([h.X.values[m][::8], h.Y.values[m][::8]]))",
           "_p = np.column_stack([h.X.values[m][::8], h.Y.values[m][::8]])\n"
           "        pts.append(_p)\n"
           "        widp.extend([w] * len(_p))"),
    ('A3', "wid = [''] * n_train",
           "wid = list(widp)"),
    ('A4', "P = np.vstack(pts); Uv = np.concatenate(us)",
           "P = np.vstack(pts); Uv = np.concatenate(us)\n"
           "    globals()['_UFIELD_PTS'] = P"),
]

_FAILED = []
for _tag, _old, _new in _ANCHORS:
    _n = _src.count(_old)
    print('  anchor %s: %d match(es)' % (_tag, _n))
    if _n != 1:
        _FAILED.append((_tag, _old, _n))

if _FAILED:
    print('\n' + '=' * 72)
    print('ANCHOR MISMATCH — patch NOT applied, nothing was modified.')
    print('Your v22 source differs from what this patch expects. Full source')
    print('below; send it over and the anchors get re-cut against your text.')
    print('=' * 72)
    for _tag, _old, _n in _FAILED:
        print('  %s expected 1 match, found %d, for:\n    %r' % (_tag, _n, _old))
    print('-' * 72)
    print(_src0)
    raise SystemExit('P1 aborted: anchor mismatch')

for _tag, _old, _new in _ANCHORS:
    _src = _src.replace(_old, _new)

ast.parse(_src)                      # syntax gate before exec
exec(compile(_src, '<build_ufield_patched>', 'exec'), globals())

print('\nbuild_ufield AFTER  sha256 =',
      hashlib.sha256(inspect.getsource(build_ufield).encode()).hexdigest())
print('PATCH P1 APPLIED — field points now carry per-point well ids')
print('\n--- patched source (eyeball the four edits) ---')
print(_src)
_PATCH_APPLIED.add('P1'); globals()['_PATCH_APPLIED'] = _PATCH_APPLIED


## P2 — exclude-then-build self-exclusion

This is the subtle one, and it's why the obvious fix fails.

A well sits *on* its own dense lateral. With `field_sub=8` a well contributes
roughly 820 field points spaced ~8 ft apart, spanning ±160 ft around any station
on it. So for any query point on that lateral, **all 40 nearest neighbours are the
well's own points** — the next well is hundreds of feet away.

Masking those neighbours *after* the k-nearest query therefore zeroes every weight
→ `s ≈ 0` → `est = NaN` → the `okk.sum() < 50` guard trips → `field_blend` returns
`pred` untouched. You wouldn't get a leak. You'd get **v22 with the field blend
silently switched off**, scoring maybe 7.5–8 instead of ~6.55, and the join would
compare the BiGRU against a crippled v22.

The fix is to rebuild the KD-tree *without* the well's points, then query. The
current well arrives via the module-global `_CUR_WELL` (set by the P3 wrapper), so
no call site changes — and an explicit `self_well` argument always wins, so this
is correct whether or not v22 already threads it through.

The point array is read from `_UFIELD['pts']` when present, falling back to the
`_UFIELD_PTS` global that P1 sets. The tree cache holds exactly one entry: each
tree spans the full field (~600k points), so caching several would cost hundreds
of MB for no benefit when wells are processed one at a time.

In [ ]:
# ===== P2: exclude-then-build field query =====
from scipy.spatial import cKDTree

_CUR_WELL = globals().get('_CUR_WELL', None)   # set by the P3 wrapper
_EXCL_CACHE = {}

def _field_points():
    """Raw field point array: _UFIELD['pts'] if present, else the P1 global."""
    F = _UFIELD
    pts = F.get('pts') if isinstance(F, dict) else None
    if pts is None:
        pts = globals().get('_UFIELD_PTS')
    if pts is None:
        raise RuntimeError('field point array missing — re-run P1 then P4')
    return np.asarray(pts)


def _excluded_field(self_well):
    """Return (tree, U) with self_well's own points removed.

    Masking neighbours AFTER a shared-tree query is wrong: a well sits on its own
    dense lateral, so all k neighbours are its own points and masking zeroes every
    weight, silently disabling the blend. Rebuild without them instead.
    """
    F = _UFIELD
    if self_well is None:
        return F['tree'], F['U']
    hit = _EXCL_CACHE.get(self_well)
    if hit is not None:
        return hit
    keep = np.asarray(F['wid']) != self_well
    if keep.all():                      # test wells aren't in the field: no-op
        out = (F['tree'], F['U'])
    else:
        out = (cKDTree(_field_points()[keep]), np.asarray(F['U'])[keep])
    _EXCL_CACHE.clear()                 # only the current well is ever needed
    _EXCL_CACHE[self_well] = out
    return out


def _field_query(xq, yq, self_well=None):
    F = _UFIELD
    k = int(CONFIG.get('field_k', 40))
    soft = float(CONFIG.get('field_soft', 400.0))
    if self_well is None:               # explicit argument always wins
        self_well = globals().get('_CUR_WELL', None)
    tree, U = _excluded_field(self_well)
    dd, idx = tree.query(np.column_stack([xq, yq]), k=k)
    wgt = 1.0 / (dd + soft) ** 2
    s = wgt.sum(1)
    est = np.einsum('nk,nk->n', wgt, U[idx]) / np.maximum(s, 1e-12)
    var = np.einsum('nk,nk->n', wgt,
                    (U[idx] - est[:, None]) ** 2) / np.maximum(s, 1e-12)
    est[s <= 1e-12] = np.nan
    return est, dd[:, 0], np.sqrt(np.maximum(var, 0))

print('PATCH P2 APPLIED — _field_query excludes-then-builds, driven by _CUR_WELL')
_PATCH_APPLIED.add('P2'); globals()['_PATCH_APPLIED'] = _PATCH_APPLIED


## P3 — branch spread + current-well plumbing

Two wrappers, both idempotent (guarded on `_orig_*` already existing, so
re-running cannot double-wrap).

**`field_blend`** — v22 already builds `arr['_branch_paths']`, the per-branch
predictions whose disagreement drives the inverse-variance fusion. That spread
*is* the candidate routing signal. The wrapper records the mean over blind
stations of the median-absolute-deviation across branch paths into
`diag['branch_spread_mean']`, then defers to the original.

The blind mask is resolved defensively: `arr['blind']` first (v22's canonical
definition, and the branch paths live in `arr` space), falling back to
`h.TVT_input.isna()` only if the lengths say `arr` space and `h` space agree. If
neither lines up, the diagnostic is skipped rather than computed on a misaligned
mask — a wrong routing variable is worse than a missing one.

**`predict_well_diag`** — sets `_CUR_WELL` from `h.attrs['well']` for the duration
of the call, in `try/finally` so it's always restored, including on exceptions.

In [ ]:
# ===== P3: field_blend spread capture + predict_well_diag well plumbing =====

def _resolve_blind(h, arr, n):
    """Blind mask of length n, or None if no aligned mask can be found."""
    b = arr.get('blind') if isinstance(arr, dict) else None
    if b is not None and len(b) == n:
        return np.asarray(b, dtype=bool)
    try:
        hb = ~np.isfinite(h['TVT_input'].values.astype(float))
        if len(hb) == n:
            return hb
    except Exception:
        pass
    return None


if '_orig_field_blend' not in globals():
    _orig_field_blend = field_blend

def field_blend(*a, **k):
    """Record per-well branch spread into diag, then defer to v22's original."""
    h = k.get('h', a[0] if len(a) >= 1 else None)
    diag = k.get('diag', a[2] if len(a) >= 3 else None)
    arr = k.get('arr', a[3] if len(a) >= 4 else None)
    try:
        bp_list = arr.get('_branch_paths') if isinstance(arr, dict) else None
        if isinstance(diag, dict) and bp_list is not None and len(bp_list) >= 2:
            bp = np.stack(bp_list)
            sp = np.median(np.abs(bp - np.median(bp, axis=0)), axis=0)
            blind = _resolve_blind(h, arr, len(sp))
            if blind is not None:
                v = sp[blind]
                v = v[np.isfinite(v)]
                if len(v):
                    diag['branch_spread_mean'] = round(float(v.mean()), 3)
                    diag['branch_spread_n'] = int(len(v))
    except Exception:
        pass                              # a diagnostic must never break a prediction
    return _orig_field_blend(*a, **k)


if '_orig_predict_well_diag' not in globals():
    _orig_predict_well_diag = predict_well_diag

def predict_well_diag(h, t, *a, **k):
    """Publish the current well to _CUR_WELL so _field_query can self-exclude."""
    global _CUR_WELL
    _prev = globals().get('_CUR_WELL', None)
    try:
        _CUR_WELL = h.attrs.get('well') if hasattr(h, 'attrs') else None
    except Exception:
        _CUR_WELL = None
    try:
        return _orig_predict_well_diag(h, t, *a, **k)
    finally:
        _CUR_WELL = _prev

print('PATCH P3 APPLIED — branch_spread_mean captured; _CUR_WELL plumbed')
print('  wrapped field_blend       :', _orig_field_blend)
print('  wrapped predict_well_diag :', _orig_predict_well_diag)
_PATCH_APPLIED.add('P3'); globals()['_PATCH_APPLIED'] = _PATCH_APPLIED


## P4 — rebuild the field

The field must be rebuilt so `_UFIELD['wid']` picks up the per-point well tags and
the point array gets populated. Without this, P2 has nothing to exclude against.

In [ ]:
# ===== P4: rebuild the structural field with tagged well ids =====
import time
assert not CONFIG.get('field_aug', False), 'field_aug flipped True since P0'

_t0 = time.time()
_UFIELD = None
F = build_ufield()
print('field rebuilt in %.0fs' % (time.time() - _t0))

_wid = np.asarray(_UFIELD['wid'])
_n_unique = len(np.unique(_wid))
_pts = _field_points()
print('field points      : %d' % len(_UFIELD['U']))
print('unique well tags  : %d' % _n_unique)
print('point array shape : %s  (source: %s)'
      % (_pts.shape, 'ufield["pts"]' if _UFIELD.get('pts') is not None
                     else '_UFIELD_PTS global'))

assert _n_unique > 100, (
    'wid tagging did not take effect (%d unique tags). Re-run P1, then P4.' % _n_unique)
assert len(_pts) == len(_UFIELD['U']) == len(_wid), 'field arrays misaligned'
_EXCL_CACHE.clear()
print('\nP4 OK — field carries per-point well ids')


## P5 — validation gate

This decides whether anything below is worth reading. It runs the field query on
one training well twice — self-exclusion off, then on — and reports nearest-
neighbour distance and field-only blind MAE for each.

| | nn_dist (median) | field-only blind MAE |
|---|---|---|
| self-**included** (leaked) | ~2 ft | ~0.2–0.4 |
| self-**excluded** (real) | hundreds of ft | ~13–15 |

The leaked MAE being near zero is the point: the well is reading its own answer
off its own lateral. If the two rows look alike, exclusion isn't firing.

The third check matters as much: the excluded estimate must stay **finite**.
All-NaN is the tree-rebuilt-empty failure that silently switches the blend off.

In [ ]:
# ===== P5: VALIDATION GATE — prove self-exclusion is firing =====
_tw = wells('train')
_w = _tw[100] if len(_tw) > 100 else _tw[0]
_h, _t = load_well('train', _w)
_h.attrs['well'] = _w

_z = _h['Z'].values.astype(float)
_tvt = _h['TVT'].values.astype(float)
_tin = _h['TVT_input'].values.astype(float)
_blind = ~np.isfinite(_tin)
_bok = _blind & np.isfinite(_tvt)
_ku = (~_blind) & np.isfinite(_tin)

print('gate well: %s   blind stations: %d   known: %d\n' % (_w, _bok.sum(), _ku.sum()))
print('%-16s %12s %12s %10s' % ('mode', 'nn_dist(med)', 'blind MAE', 'finite%'))
print('-' * 54)

_res = {}
for _tag, _sw in [('self-included', None), ('self-excluded', _w)]:
    _EXCL_CACHE.clear()
    _est, _dmin, _ = _field_query(_h.X.values, _h.Y.values, self_well=_sw)
    _fin = np.isfinite(_est)
    _ok = _ku & _fin
    if _ok.sum() < 10:
        print('%-16s %12s %12s %9.1f%%' % (_tag, 'n/a', 'NO ANCHOR', 100 * _fin.mean()))
        _res[_tag] = (np.nan, np.nan); continue
    _off = np.median((_tin + _z)[_ok] - _est[_ok])
    _ftvt = _est + _off - _z
    _m = _bok & np.isfinite(_ftvt)
    _res[_tag] = (float(np.median(_dmin[_bok])),
                  float(np.abs(_ftvt[_m] - _tvt[_m]).mean()))
    print('%-16s %12.0f %12.2f %9.1f%%' % (_tag, _res[_tag][0], _res[_tag][1],
                                           100 * _fin.mean()))

_EXCL_CACHE.clear()
_nn_in, _mae_in = _res['self-included']
_nn_ex, _mae_ex = _res['self-excluded']

print('\n' + '=' * 54)
_pass = True
if not np.isfinite(_mae_ex):
    print('FAIL: excluded query produced no usable anchor (all-NaN field).')
    print('      This is the "field blend silently off" failure mode.')
    _pass = False
if np.isfinite(_nn_ex) and np.isfinite(_nn_in) and _nn_ex < 10 * max(_nn_in, 1.0):
    print('FAIL: nn_dist barely moved (%.0f -> %.0f). Exclusion is not firing.'
          % (_nn_in, _nn_ex))
    _pass = False
if np.isfinite(_mae_ex) and _mae_ex < 3.0:
    print('FAIL: excluded blind MAE %.2f implausibly low — a leak survived.' % _mae_ex)
    _pass = False
if _pass:
    print('GATE PASSED — self-exclusion is real.')
    print('  leaked MAE %.2f (nn %.0f ft)  ->  honest MAE %.2f (nn %.0f ft)'
          % (_mae_in, _nn_in, _mae_ex, _nn_ex))
    print('  Honest MAE should sit near 13-15. Proceed to P6.')
else:
    print('\nDO NOT RUN P6 until this passes.')
print('=' * 54)


## P6 — the OOF loop

Scores v22 over the training wells with each well excluded from its own field,
mirroring deployment exactly: a test well is never in the field.

`v22_oof.pkl` carries six things, keyed by well:

- `err` — per-well blind-zone MAE in **absolute TVT space**
- `pred` — blind station indices and predicted TVT, for the join and any offline blend
- `spread` — `branch_spread_mean`, routing candidate #1
- `nn` — `nn_dist`, field-support distance, routing candidate #2
- `field_conf` — routing candidate #3
- `status` — which code path each well took, so failures can be clustered

**The comparability trap.** The BiGRU trains on the increment target with its own
per-well datum and `u_last`; v22 has its own anchor logic. They're only comparable
if both errors are `abs(pred_TVT − true_TVT)` over the *identical* blind mask of
the *identical* wells. This loop is in absolute TVT space for exactly that reason
— make sure the BiGRU side is too, or you'll chase a phantom.

**Validity gate.** Final OOF MAE should land near **~6.5**. Around 7.5–8 means the
field blend is off. Well below 6 means a leak survived.

Budget 45–75 min — roughly 3.6 s/well plus one tree rebuild per well.
Checkpoints every 25 wells, so a timeout doesn't cost the run.

In [ ]:
# ===== P6: v22 OOF over the training wells (leak-free via self-exclusion) =====
import pickle, time, traceback
from collections import Counter

assert {'P1', 'P2', 'P3'} <= _PATCH_APPLIED, \
    'run P1-P3 first (applied: %s)' % sorted(_PATCH_APPLIED)

tr_wells = wells('train')
v22_err, v22_pred, v22_spread, v22_nn = {}, {}, {}, {}
v22_fc, v22_status = {}, {}
skipped = []
t0 = time.time()

def _save(partial):
    pickle.dump({'err': v22_err, 'pred': v22_pred, 'spread': v22_spread,
                 'nn': v22_nn, 'field_conf': v22_fc, 'status': v22_status,
                 'skipped': skipped, 'partial': partial},
                open('v22_oof.pkl', 'wb'))

for i, w in enumerate(tr_wells):
    try:
        h, t = load_well('train', w)
        h.attrs['well'] = w                    # -> _CUR_WELL -> self-exclusion active

        truth = h['TVT'].values.astype(float)
        blind = ~np.isfinite(h['TVT_input'].values.astype(float))
        m = blind & np.isfinite(truth)
        if m.sum() < 1:
            skipped.append((w, 'no scorable blind stations'))
            v22_status[w] = 'skip_no_blind'
            continue

        pred, status, diag = predict_well_diag(h, t)
        pred = np.asarray(pred, dtype=float)

        good = m & np.isfinite(pred)
        if good.sum() < 1:
            skipped.append((w, 'all predictions NaN'))
            v22_status[w] = 'skip_all_nan'
            continue

        v22_err[w] = float(np.abs(pred[good] - truth[good]).mean())
        v22_pred[w] = dict(blind_idx=np.where(good)[0].astype(np.int32),
                           tvt_hat=pred[good].astype(np.float32))
        v22_spread[w] = float(diag.get('branch_spread_mean', np.nan))
        v22_nn[w] = float(diag.get('nn_dist', np.nan))
        v22_fc[w] = float(diag.get('field_conf', np.nan))
        v22_status[w] = str(status)

    except Exception as e:
        skipped.append((w, repr(e)[:120]))
        v22_status[w] = 'error_' + type(e).__name__
        if len(skipped) <= 3:
            traceback.print_exc()

    if (i + 1) % 25 == 0:
        _run = np.mean(list(v22_err.values())) if v22_err else np.nan
        _el = time.time() - t0
        print('%4d/%d  running mean err %.3f  skipped %d  [%.0fs, ~%.0fs left]'
              % (i + 1, len(tr_wells), _run, len(skipped), _el,
                 _el / (i + 1) * (len(tr_wells) - i - 1)), flush=True)
        _save(True)

_save(False)

_errs = np.array(list(v22_err.values()))
_oof = float(_errs.mean())
_sp = np.array([v22_spread[w] for w in v22_err])
_nn = np.array([v22_nn[w] for w in v22_err])
_fc = np.array([v22_fc[w] for w in v22_err])

print('\n' + '=' * 66)
print('saved v22_oof.pkl | v22 OOF MAE %.3f over %d wells (%.0f min)'
      % (_oof, len(v22_err), (time.time() - t0) / 60))
print('  per-well err: p10 %.2f  median %.2f  p90 %.2f  max %.2f'
      % tuple(np.percentile(_errs, [10, 50, 90]).tolist() + [_errs.max()]))
print('  branch_spread present : %.1f%%' % (100 * np.isfinite(_sp).mean()))
print('  nn_dist present       : %.1f%%' % (100 * np.isfinite(_nn).mean()))
print('  field_conf present    : %.1f%%' % (100 * np.isfinite(_fc).mean()))
print('  statuses:', Counter(v22_status.values()).most_common(6))
if skipped:
    print('  skipped %d wells; first few: %s' % (len(skipped), skipped[:5]))

print('-' * 66)
if 6.0 <= _oof <= 7.2:
    print('VALIDITY GATE PASSED — %.3f is in range of v22\'s known local ~6.5.' % _oof)
elif _oof > 7.2:
    print('GATE FAILED (%.3f too high) — the field blend is likely switched off.' % _oof)
    print('  Check P5 passed and that P1->P4 ran in order.')
else:
    print('GATE FAILED (%.3f too low) — a leak likely survived. Re-check P5.' % _oof)

if np.isfinite(_fc).mean() < 0.80:
    print('\nNOTE: field_conf present on only %.1f%% of wells. If your v22 diag')
    print('  never had a "field_conf" key this is expected and harmless — the')
    print('  other two routing candidates are unaffected. If it DOES have one,')
    print('  this says the field disengaged on most wells; investigate before')
    print('  trusting the join.')
print('=' * 66)
print('\nDownload v22_oof.pkl from the Output tab, then run the join against')
print('bigru_oof.pkl. Treat anything under ~0.2 local improvement as noise.')


## Notes

**Do not run v22's submission cell in this fork.** `field_blend` and
`predict_well_diag` are wrapped and `build_ufield` is rewritten in memory. The
wrappers are behaviour-preserving on test wells (`_CUR_WELL` is `None`,
`keep.all()` is True, so the query is bit-identical), but there's no reason to
take the risk on a scoring run. Keep the real v22 sealed.

**Re-running is safe.** P1 asserts exactly-once anchors before touching anything.
P2 is a plain redefinition. P3 guards on `_orig_*` so it can't double-wrap. P4
clears `_EXCL_CACHE`.

**On `field_conf`.** It's captured via `diag.get('field_conf', np.nan)`, so if
your v22 doesn't produce that key the value is simply NaN and nothing breaks. The
P6 note distinguishes the two readings of a low presence rate — key absent
(harmless) versus field disengaging (not harmless) — because they look identical
in the output and mean opposite things.

**Reading the join.** Across v19/v22/v23/v24 the local and leaderboard orderings
came out perfectly inverted, with spreads of 0.04 local and 0.02 LB. That's almost
certainly noise rather than genuine anti-correlation, but the implication holds
either way: local differences at the 0.04 scale carry no predictive signal for the
leaderboard. Don't ship on a 0.05.

**One asymmetry, and it favours you.** The BiGRU's OOF is a true 5-fold held-out
number. v22's OOF here benefits from a field built over all training wells minus
self, which is slightly more favourable to v22. So if a blend wins in this
comparison, it should win by at least that much on the leaderboard, not less.